# 2026-11-04-bt-FeSe-ob002-MvsT-Fw-Bokeh

## Imports, Aesthetics and Bokeh Setup

In [15]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [33]:
# ========================================
# CODE IMPORTS
# ========================================
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import os
import statistics

# ========================================
# PLOT IMPORTS
# ========================================
from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, Span
from bokeh.palettes import Category20_20, Category10_10

output_notebook()


# ========================================
# FUNCTION IMPORTS
# ========================================
sys.path.insert(1, './..')
import functions
from functions import read_squid_file, determine_critical_temp, determine_critical_field

Loading BokehJS ...

## Data Reading

In [17]:
print("FIELD WARMING")
print("="*40)

from pathlib import Path
runs_fw = {}
pathlist = Path(
    "../../data/2026/10/FeSe_bo001_MvsT_FW_VariousFields"
).glob("**/*.dat")
for path in sorted(pathlist):
    run_number = path.stem[-5:]
    run_number = int(run_number) - 1
    runs_fw[run_number] = read_squid_file(str(path))
for run_number, data in runs_fw.items():
    print(f"Run {run_number}")

FIELD WARMING
Run 0
Run 1
Run 2
Run 3
Run 4


## Plotting a Raw data Line in Bokeh

In [29]:

p = figure(width=800,
           height=450,
           title = "261004 - FeSe ob002 FW",
           title_location = "above",
           x_axis_label="Temperature (K)",
           y_axis_label="DC Moment Fixed Ctr (emu)",
           )


for i in range(len(runs_fw)):
    p.line(runs_fw[i]["Temperature (K)"],
           runs_fw[i]["DC Moment Fixed Ctr (emu)"],
           line_color=Category20_20[2*i])


p.add_tools(HoverTool(tooltips=[("Temperature (K)", "@x{0.00}"),
                                ("DC Moment (emu)", "@y{0.000e}")]))
show(p)

## Add fits and plot this

In [36]:

p = figure(width=800,
           height=450,
           title = "261004 - FeSe ob002 FW",
           title_location = "above",
           x_axis_label="Temperature (K)",
           y_axis_label="DC Moment Fixed Ctr (emu)",
           )


for i in range(len(runs_fw)):
    # Reading in and calculating fit

    # make dictionary field for the critical temperature and fit
    fits = {}
    fits["crit_temp"], fits["fit"] = determine_critical_temp(runs_fw[i])


    # plotting
    p.line(runs_fw[i]["Temperature (K)"],
           runs_fw[i]["DC Moment Fixed Ctr (emu)"],
           line_color=Category20_20[2*i])


    p.line(runs_fw[i]["crit_temp"])


p.add_tools(HoverTool(tooltips=[("Temperature (K)", "@x{0.00}"),
                                ("DC Moment (emu)", "@y{0.000e}")]))
show(p)

print(fits.items())

dict_items([('crit_temp', np.float64(8.384964498752012)), ('fit', {'normal_mask': array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       F

In [37]:
from bokeh.plotting import figure, show
from bokeh.models import HoverTool
from bokeh.palettes import Viridis256

p = figure(width=800, height=450,
           title="261104 - FeSe ob002 FW",
           x_axis_label="Temperature (K)",
           y_axis_label="DC Moment Fixed Ctr (emu)")

n_runs = len(runs_fw)
palette = [Viridis256[int(k * 255 / max(n_runs - 1, 1))] for k in range(n_runs)]

results = {}          # keep Tc and fit parameters out of the dataframes
data_renderers = []   # hover applies to these only

for colour, (run_number, df) in zip(palette, runs_fw.items()):
    t = df["Temperature (K)"].to_numpy(dtype=float)
    m = df["DC Moment Fixed Ctr (emu)"].to_numpy(dtype=float)
    field = df["Magnetic Field (Oe)"].mean()

    tc, fit = determine_critical_temp(df)
    results[run_number] = {"tc": tc, "fit": fit, "field": field}

    label = f"{field:.0f} Oe, Tc = {tc:.2f} K" if np.isfinite(tc) else f"{field:.0f} Oe, no fit"

    # Data
    data_renderers.append(
        p.line(t, m, line_color=colour, line_width=1.5, legend_label=label)
    )

    if not np.isfinite(tc):
        continue

    # Fit lines, drawn only where they fall inside the measured moment range
    t_line = np.linspace(t.min(), t.max(), 400)
    for key, dash in (("normal", "dashed"), ("trans", "dotted")):
        slope, intercept = fit[key]
        m_line = slope * t_line + intercept
        keep = (m_line >= m.min()) & (m_line <= m.max())
        p.line(t_line[keep], m_line[keep], line_color=colour,
               line_dash=dash, line_width=1, legend_label=label)

    # Tc marker, sitting on the intersection of the two lines
    n_slope, n_int = fit["normal"]
    p.scatter([tc], [n_slope * tc + n_int], size=9, color=colour,
              line_color="black", legend_label=label)

p.add_tools(HoverTool(
    renderers=data_renderers,
    tooltips=[("T (K)", "@x{%.3f}"), ("Moment (emu)", "@y{%.3e}")],
    formatters={"@x": "printf", "@y": "printf"},
))
p.legend.click_policy = "hide"   # one click hides data, fits and Tc marker for that run
p.legend.location = "top_left"
show(p)